# Stretch: making the pipeline sturdier, worked solutions

The answers to the stretch notebook. Try each one before looking.

## Setup

In [ ]:
import os
import sqlite3
import sys
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.insert(0, "sessions/session-10/demos")
import genre_pipeline as gp

raw = gp.load_messy(gp.MESSY)
plays = gp.clean_plays(raw)
clean = pd.read_csv("data/clean/plays.csv")
print(len(raw), "raw rows,", len(plays), "cleaned")

## Stretch 1: swap the source

In [ ]:
def load_from_db(path):
    """Return every play from the database, with the columns of data/clean/plays.csv."""
    query = """
        SELECT p.play_id, p.played_at, a.artist_name, p.track_name,
               a.genre, a.country, p.minutes_played, p.device, p.skipped
        FROM plays AS p
        JOIN artists AS a ON a.artist_id = p.artist_id
        ORDER BY p.play_id
    """
    con = sqlite3.connect(path)
    plays = pd.read_sql(query, con)
    con.close()
    return plays


from_db = load_from_db("data/music.db")
assert from_db.shape == (2183, 9)
assert list(from_db.columns) == list(clean.columns)
assert from_db.dtypes.equals(clean.dtypes)

In [ ]:
from_db_summary = gp.summarise_by_genre(from_db)
messy_summary = gp.summarise_by_genre(plays)

assert from_db_summary.loc[0, "minutes"] == 2874.5
assert messy_summary.loc[0, "minutes"] == 2753.2
from_db_summary.head(3)

The database says 2,874.5 minutes for Electronic; today's pipeline said
2,753.2. The database is right: it was never damaged. The pipeline's
number is lower because 94 plays lost their minutes in the messy file and
we chose to leave them missing.

The point of the exercise is the shape, though. **One stage was swapped and
nothing else changed.** `summarise_by_genre` has no idea where its data
came from, and it does not need to. The cleaning stage simply drops out,
because the database does not need it. That is what keeping the stages
apart buys you, and why your project can start on a CSV and move to a
database later without a rewrite.

The `JOIN` matches each play to exactly one artist, so the row count stays
at 2,183: no fan-out. The `assert` on the shape is the session 7 habit of
checking the row count after every join.

## Stretch 2: a cleaning report

In [ ]:
def cleaning_report(raw, plays):
    """Return a dictionary describing what cleaning did to the data."""
    return {
        "rows in": len(raw),
        "duplicates removed": int(raw.duplicated().sum()),
        "rows out": len(plays),
        "minutes missing": int(plays["minutes_played"].isna().sum()),
        "devices unknown": int((plays["device"] == "unknown").sum()),
    }


report = cleaning_report(raw, plays)
assert report["duplicates removed"] == 40
assert report["minutes missing"] == 94
assert report["devices unknown"] == 69
assert report["rows in"] - report["duplicates removed"] == report["rows out"]

In [ ]:
for name, number in report.items():
    print(f"  {name:20} {number:>5}")

The report returns a dictionary and the loop at the end prints it, so the
same report could go into a file or a check instead. `int(...)` turns
pandas' own number types into plain Python ones, which print and compare
without surprises.

The last `assert` is the useful one: rows in minus duplicates should be
rows out. If a future change to `clean_plays` started dropping rows for
some other reason, this line would be the first to notice.

## Stretch 3: fail early, with a clear message

Fed the wrong file, `clean_plays` fails like this:

In [ ]:
try:
    gp.clean_plays(gp.load_messy("data/clean/artists.csv"))
except KeyError as error:
    print("KeyError:", error)

True, and it sounds like a bug in the code: what is wrong with
`play_id`? The real problem is that this is the wrong file altogether.

In [ ]:
REQUIRED = ["play_id", "played_at", "artist_name", "genre", "country",
            "device", "skipped", "minutes played "]


def check_columns(df, required):
    """Return nothing if df has every required column; raise ValueError if not."""
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"the input file is missing these columns: {missing}")


check_columns(raw, REQUIRED)
print("the messy file has every column")

try:
    check_columns(gp.load_messy("data/clean/artists.csv"), REQUIRED)
except ValueError as error:
    print("ValueError:", error)

Now the message names the problem: five of the columns the pipeline needs
are not in this file, so it is not a plays file at all. In a script the
check goes straight after `load`, before anything has been calculated or
written.

**Why crash at all?** A pipeline that carries on with the wrong input
writes a wrong CSV and a confident chart, and nobody goes looking. A clear
crash is read by the one person who can fix it, at the moment it happens.
The person reading it is usually you, three months from now, having
forgotten everything.

(`try` and `except` here only let the notebook show the error and carry
on. Catching errors properly is session 12.)

## Stretch 4: one year, from the command line

In [ ]:
def parse_year(argv):
    """Return the year given after the script name as a number, or None."""
    if len(argv) < 2:
        return None
    return int(argv[1])


def plays_in_year(plays, year):
    """Return only the plays from year, or all of them when year is None."""
    if year is None:
        return plays
    return plays[plays["played_at"].str.startswith(str(year))]


assert parse_year(["pipeline.py"]) is None
assert parse_year(["pipeline.py", "2025"]) == 2025
assert len(plays_in_year(plays, None)) == 2183
assert len(plays_in_year(plays, 2025)) == 1061
gp.summarise_by_genre(plays_in_year(plays, 2025)).head(3)

**Why hand `parse_year` the list?** Because then it can be checked with
any list you like, as above, without running a script. A function that
read `sys.argv` itself could only be tested from a terminal. It is the same
rule as everywhere today: take inputs as parameters, return results.

In the script, `main()` would begin like this, and the output file names
could carry the year so one run does not overwrite another:

```python
year = parse_year(sys.argv)
plays = plays_in_year(clean_plays(load_messy(MESSY)), year)
```

For more than one option, Python's `argparse` module does this properly,
with `--help` for free. It is worth looking up once you need it.

## Stretch 5: a cleaning decision as a parameter

In [ ]:
def fill_minutes(plays, how="keep"):
    """Return a copy of plays with missing minutes kept, dropped, or set to the median."""
    if how == "keep":
        return plays.copy()
    if how == "drop":
        return plays.dropna(subset=["minutes_played"])
    if how == "median":
        filled = plays.copy()
        median = filled["minutes_played"].median()
        filled["minutes_played"] = filled["minutes_played"].fillna(median)
        return filled
    raise ValueError(f"how must be 'keep', 'drop' or 'median', not {how!r}")


assert len(fill_minutes(plays, "drop")) == 2089
assert fill_minutes(plays, "median")["minutes_played"].isna().sum() == 0

In [ ]:
rows = []
for how in ["keep", "drop", "median"]:
    version = fill_minutes(plays, how)
    top = gp.summarise_by_genre(version).iloc[0]
    rows.append({"how": how,
                 "electronic plays": top["plays"],
                 "electronic minutes": top["minutes"],
                 "all minutes": round(version["minutes_played"].sum(), 1)})

rows.append({"how": "truth (database)",
             "electronic plays": from_db_summary.loc[0, "plays"],
             "electronic minutes": from_db_summary.loc[0, "minutes"],
             "all minutes": round(from_db["minutes_played"].sum(), 1)})
pd.DataFrame(rows)

Three honest-looking answers:

* **keep** gets the play counts right (547) and the minutes too low,
  because 94 plays count as zero minutes.
* **drop** gets the same minutes and now the play counts are wrong too:
  2,089 plays instead of 2,183. Every skip rate changes slightly, because
  94 real plays have vanished from them.
* **median** fills each gap with 4.1 minutes and gets the overall total
  closest to the truth: 8,993.4 against 8,980.4. But 94 of its numbers are
  invented, and Electronic is still 31.1 minutes short.

Closest is not the same as right. The median version makes the chart claim
precision it does not have. Which to choose depends on the question: for
"how many times did I play each genre", keep; for "how long", keep and say
that 94 plays have no minutes, or fill and say that you filled. The one
unacceptable option is choosing without saying. Making it a parameter with
a default puts the choice in plain sight in the code.

The last line of `fill_minutes` is the stretch 3 idea again: a mistyped
option like `how="mdian"` gets a clear error instead of silently doing
"keep".

In [ ]:
try:
    fill_minutes(plays, "mdian")
except ValueError as error:
    print("ValueError:", error)

## Stretch 6: a log of every run

In [ ]:
from datetime import datetime


def log_run(path, plays, summary):
    """Add one line to the run log at path: time, plays, and the top genre."""
    path.parent.mkdir(parents=True, exist_ok=True)
    is_new = not path.exists()
    with open(path, "a", encoding="utf-8") as f:
        if is_new:
            f.write("ran_at,plays,top_genre\n")
        ran_at = datetime.now().isoformat(timespec="seconds")
        f.write(f"{ran_at},{len(plays)},{summary.iloc[0]['genre']}\n")


log = Path("output/runs.csv")
summary = gp.summarise_by_genre(plays)
before = len(pd.read_csv(log)) if log.exists() else 0
log_run(log, plays, summary)
log_run(log, plays, summary)
assert len(pd.read_csv(log)) == before + 2
pd.read_csv(log).tail(3)

`"a"` appends: each run adds a line and the history stays. With `"w"` the
file would only ever hold the last run, which is no history at all.

The header goes in only when the file is new, which is why `is_new` is
worked out **before** opening: once `open(path, "a")` runs, the file exists
whether or not it did a moment ago.

What it is for: in six months, when a number looks different, the log
tells you when it changed and what else changed with it. If the play count
jumped on one date, the input file changed; if it did not, look at the
code. Real pipelines keep exactly this kind of record, usually with a
little more in each line.

This file is not a CSV you would write with `to_csv`, because `to_csv`
replaces. (It can append too, with `mode="a"` and `header=False`, which
is worth knowing once the plain version makes sense.)